# SIP-HyperGCN v4: Causal Concept Program
Thiết kế: `docs/architecture_v4.html`. Runtime: GPU. Chạy lần lượt từ trên xuống; mỗi giai đoạn lưu checkpoint lên Drive nên có thể chạy tiếp nếu Colab bị ngắt.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd /content
!if [ -d thesis ]; then git -C thesis pull; else git clone https://github.com/NguyenBaoHuy05/thesis.git; fi

/content
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 4.67 KiB | 2.33 MiB/s, done.
From https://github.com/NguyenBaoHuy05/thesis
   47a791f..4a54a32  main       -> origin/main
Updating 47a791f..4a54a32
Fast-forward
 SIP-HyperGCN/ccp_colab.ipynb      | 163 ++++++++++++++++++++++++++++++--------
 SIP-HyperGCN/scripts/train_ccp.py |  90 ++++++++++++++++++---
 2 files changed, 208 insertions(+), 45 deletions(-)


## Cấu hình

In [4]:
import os
%cd /content/thesis/SIP-HyperGCN

os.environ["H"]  = "/content/thesis/GCCR-Skateformer-TreeRule/hypergcn"
os.environ["CK"] = "/content/drive/MyDrive/BaoHuy/base/ntu60/xsub/joint/runs-124-77624.pt"
os.environ["D"]  = "/content/drive/MyDrive/BaoHuy/KLTN_Huy-Nhu/NTU60_CS.npz"
os.environ["V4"] = "/content/drive/MyDrive/BaoHuy/work_dir/ccp"      # ket qua v4 tren Drive
os.environ["COMMON"] = "--hypergcn_root {H} --checkpoint {CK} --data_path {D}".format(**os.environ)

DRIVE_DATA = "/content/drive/MyDrive/BaoHuy/KLTN_Huy-Nhu/NTU60_CS.npz"

/content/thesis/SIP-HyperGCN


In [5]:
import shutil
os.makedirs(os.path.dirname(os.environ["D"]), exist_ok=True)
if not os.path.exists(os.environ["D"]):
    shutil.copy(DRIVE_DATA, os.environ["D"])
for k in ["H", "CK", "D"]:
    print(k, os.environ[k], "OK" if os.path.exists(os.environ[k]) else "MISSING")

H /content/thesis/GCCR-Skateformer-TreeRule/hypergcn OK
CK /content/drive/MyDrive/BaoHuy/base/ntu60/xsub/joint/runs-124-77624.pt OK
D /content/drive/MyDrive/BaoHuy/KLTN_Huy-Nhu/NTU60_CS.npz OK


## 0. Smoke test (vài batch, cả 3 giai đoạn + đánh giá)
Chạy cell này trước để bắt lỗi sớm (khoảng 3-5 phút). Kết quả nằm ở `work_dir/ccp_dbg`, không đụng vào Drive.

In [ ]:
%cd /content/thesis/SIP-HyperGCN
!python scripts/train_ccp.py $COMMON --out_dir work_dir/ccp_dbg --stages 1 2 3 \
    --epochs1 1 --epochs2 1 --epochs3 1 --max_train_batches 10 --max_val_batches 3 \
    --calib_batches 5 --log_every 5 --batch_size 32
!python scripts/eval_ccp.py $COMMON --ccp_ckpt work_dir/ccp_dbg/ccp_stage3.pt --out_dir work_dir/ccp_dbg/eval \
    --max_samples 256 --cf_max_literals 1 --cf_per_literal 8 --cf_steps 5 --robust --batch_size 32

/content/thesis/SIP-HyperGCN
[load_hypergcn] using model.hypergcn_base.Model
[load_hypergcn] loaded /content/drive/MyDrive/BaoHuy/base/ntu60/xsub/joint/runs-124-77624.pt (607 tensors)
[build_feeder] split=train args={'data_path': '/content/drive/MyDrive/BaoHuy/KLTN_Huy-Nhu/NTU60_CS.npz', 'split': 'train', 'window_size': 64, 'p_interval': [0.95], 'vel': False, 'bone': False, 'debug': False, 'random_rot': False, 'random_choose': False, 'random_shift': False, 'random_move': False}
[split] train rows=40091 VAL=8014 TRAIN-FIT=32077 (val_frac=0.2, seed=0)
[bank] concepts=80 (geo=57 incl. 6 extra, latent=23)
[ccp] c6=256 before-pairs=424 params=136615
[calibrate] 41 signals over 10237 frames; tau init = median q
  hand_head            mu=+1.0486 sd=0.4178 [L]
  hand_chest           mu=+0.7511 sd=0.2567 [L]
  hands_dist           mu=+0.7841 sd=0.4787 [L]
  hand_body            mu=+0.3846 sd=0.1920 [L]
  hand_above_head      mu=-0.7963 sd=0.5144 [L]
  hand_above_shoulder  mu=-0.3903 sd=0.4997 [

## 1. Giai đoạn 1: concept (ngưỡng τ, concept ẩn)
Hiệu chỉnh tín hiệu rồi học concept theo CSV (MIL). HyperGCN đóng băng.

In [10]:
%%bash
python scripts/train_ccp.py $COMMON --out_dir $V4 --stages 1

[load_hypergcn] using model.hypergcn_base.Model
[load_hypergcn] loaded /content/drive/MyDrive/BaoHuy/base/ntu60/xsub/joint/runs-124-77624.pt (607 tensors)
[build_feeder] split=train args={'data_path': '/content/drive/MyDrive/BaoHuy/KLTN_Huy-Nhu/NTU60_CS.npz', 'split': 'train', 'window_size': 64, 'p_interval': [0.95], 'vel': False, 'bone': False, 'debug': False, 'random_rot': False, 'random_choose': False, 'random_shift': False, 'random_move': False}
[split] train rows=40091 VAL=8014 TRAIN-FIT=32077 (val_frac=0.2, seed=0)
[bank] concepts=80 (geo=57 incl. 6 extra, latent=23)
[ccp] c6=256 before-pairs=424 params=136615
[calibrate] 41 signals over 245489 frames; tau init = median q
  hand_head            mu=+1.0342 sd=0.4149 [L]
  hand_chest           mu=+0.7551 sd=0.2362 [L]
  hands_dist           mu=+0.8234 sd=0.4945 [L]
  hand_body            mu=+0.3779 sd=0.1929 [L]
  hand_above_head      mu=-0.7795 sd=0.5361 [L]
  hand_above_shoulder  mu=-0.3652 sd=0.5244 [L]
  hand_reach_fwd       mu

/content/thesis/SIP-HyperGCN/scripts/train_ccp.py:228: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  run["loss"] += float(loss) * len(y)


## 2. Giai đoạn 2: chương trình logic
Theo dõi `VAL logic=`: accuracy khi chỉ dùng tầng logic.

In [11]:
%%bash
python scripts/train_ccp.py $COMMON --out_dir $V4 --stages 2 --init_ckpt $V4/ccp_stage1.pt

[load_hypergcn] using model.hypergcn_base.Model
[load_hypergcn] loaded /content/drive/MyDrive/BaoHuy/base/ntu60/xsub/joint/runs-124-77624.pt (607 tensors)
[build_feeder] split=train args={'data_path': '/content/drive/MyDrive/BaoHuy/KLTN_Huy-Nhu/NTU60_CS.npz', 'split': 'train', 'window_size': 64, 'p_interval': [0.95], 'vel': False, 'bone': False, 'debug': False, 'random_rot': False, 'random_choose': False, 'random_shift': False, 'random_move': False}
[split] train rows=40091 VAL=8014 TRAIN-FIT=32077 (val_frac=0.2, seed=0)
[bank] concepts=80 (geo=57 incl. 6 extra, latent=23)
[ccp] c6=256 before-pairs=424 params=136615
[load] /content/drive/MyDrive/BaoHuy/work_dir/ccp/ccp_stage1.pt (stage 1, epoch 6, VAL {'logic': 0.1437484402295982, 'fused': 0.4958822061392563, 'net': 0.9888944347392064, 'concept_loss': 0.44101765089151657, 'alpha': 0.4999999701976776, 'stage': 1, 'epoch': 6, 'train_loss': 0.4406030239340312, 'train_acc': 0.1371319860279441})
==================== stage 2 ================

/content/thesis/SIP-HyperGCN/scripts/train_ccp.py:228: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  run["loss"] += float(loss) * len(y)


## 3. Giai đoạn 3: kết hợp + nhất quán phản thực tế
Fine-tune HyperGCN (lr nhỏ) với L_cf. Thêm `--freeze_backbone` nếu muốn giữ nguyên HyperGCN (khi đó L_cf tắt).

In [ ]:
%%bash
python scripts/train_ccp.py $COMMON --out_dir $V4 --stages 3 --init_ckpt $V4/ccp_stage2.pt

python3: can't open file '/content/scripts/train_ccp.py': [Errno 2] No such file or directory


CalledProcessError: Command 'b'python scripts/train_ccp.py $COMMON --out_dir $V4 --stages 3 --init_ckpt $V4/ccp_stage2.pt\n'' returned non-zero exit status 2.

## 4. Đánh giá trên TEST

In [ ]:
%%bash
python scripts/eval_ccp.py $COMMON --ccp_ckpt $V4/ccp_stage3.pt --out_dir $V4/eval --robust

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(os.path.join(os.environ["V4"], "eval", "report.md"), encoding="utf-8").read()))